### Turn alignment
#### Do prompt / response_a / response_b have the same number of turns in every row, and are there null/empty turns inside the lists?

In [ ]:
import warnings
from ast import literal_eval
import pandas as pd

warnings.filterwarnings("ignore", category=SyntaxWarning)

df = pd.read_csv("../data/raw/train.csv")

def safe_parse(x):
    try:
        parsed = literal_eval(x)
        if isinstance(parsed, list):
            return parsed
    except (ValueError, SyntaxError):
        pass
    return [x]

cols = ["prompt", "response_a", "response_b"]
for col in cols:
    df[col + "_parsed"] = df[col].apply(safe_parse)

n_p = df["prompt_parsed"].apply(len)
n_a = df["response_a_parsed"].apply(len)
n_b = df["response_b_parsed"].apply(len)

aligned = (n_p == n_a) & (n_p == n_b)
print(f"Aligned rows:    {aligned.sum()} / {len(df)}")
print(f"Misaligned rows: {(~aligned).sum()}")

def has_bad_item(lst):
    return any(item is None or (isinstance(item, str) and not item.strip()) for item in lst)

for col in cols:
    print(
        f"{col:11s} | raw contains 'null': {df[col].str.contains('null', regex=False).sum():5d}"
        f" | parsed has None/empty turn: {df[col + '_parsed'].apply(has_bad_item).sum():5d}"
    )

Aligned rows:    57335 / 57477
Misaligned rows: 142
prompt      | raw contains 'null':   176 | parsed has None/empty turn:     5
response_a  | raw contains 'null':   429 | parsed has None/empty turn:    22
response_b  | raw contains 'null':   449 | parsed has None/empty turn:    23


### Root cause of misalignment
#### Are the 142 misaligned rows caused by parse failures (null), or by genuinely different list lengths? Does json.loads fix them?

In [ ]:
import json

def le_failed(raw):
    try:
        return not isinstance(literal_eval(raw), list)
    except (ValueError, SyntaxError):
        return True

def json_parse(raw):
    try:
        parsed = json.loads(raw)
        return parsed if isinstance(parsed, list) else None
    except json.JSONDecodeError:
        return None

for col in cols:
    df[col + "_lefail"] = df[col].apply(le_failed)
    df[col + "_json"] = df[col].apply(json_parse)

print("literal_eval failures per column:")
print(df[[c + "_lefail" for c in cols]].sum(), "\n")

print("json.loads failures per column:")
print(pd.Series({c: df[c + "_json"].isna().sum() for c in cols}), "\n")

mis = ~aligned
le_any = df[[c + "_lefail" for c in cols]].any(axis=1)
print(f"Misaligned (literal_eval): {mis.sum()} | of them with a literal_eval failure: {(mis & le_any).sum()}")

ok = df[[c + "_json" for c in cols]].notna().all(axis=1)
lens = pd.DataFrame({c: df.loc[ok, c + "_json"].apply(len) for c in cols})
aligned_json = lens.nunique(axis=1) == 1
print(f"json.loads: aligned {aligned_json.sum()} / {ok.sum()} parsed rows")

literal_eval failures per column:
prompt_lefail          0
response_a_lefail    111
response_b_lefail    120
dtype: int64 

json.loads failures per column:
prompt        0
response_a    0
response_b    0
dtype: int64 

Misaligned (literal_eval): 142 | of them with a literal_eval failure: 142
json.loads: aligned 57477 / 57477 parsed rows


### Null / empty turns (after json.loads)
#### How many turns are None or empty, where do they sit, and do they correlate with the label?

In [ ]:
def bad_idx(lst):
    return [i for i, t in enumerate(lst) if t is None or (isinstance(t, str) and not t.strip())]

for col in cols:
    df[col + "_bad"] = df[col + "_json"].apply(bad_idx)

summary = {}
for col in cols:
    bad = df[col + "_bad"]
    n_none = df[col + "_json"].apply(lambda l: sum(t is None for t in l)).sum()
    n_empty = df[col + "_json"].apply(lambda l: sum(isinstance(t, str) and not t.strip() for t in l)).sum()
    last_bad = df.apply(
        lambda r: bool(r[col + "_bad"]) and r[col + "_bad"][-1] == len(r[col + "_json"]) - 1, axis=1
    ).sum()
    summary[col] = {
        "rows_with_bad": int((bad.apply(len) > 0).sum()),
        "None_turns": int(n_none),
        "empty_turns": int(n_empty),
        "rows_where_last_turn_bad": int(last_bad),
    }
print(pd.DataFrame(summary).T, "\n")

any_bad = df[[c + "_bad" for c in cols]].apply(lambda r: any(len(x) for x in r), axis=1)
print(f"Rows with any bad turn: {any_bad.sum()} ({any_bad.mean():.2%})\n")

label = df[["winner_model_a", "winner_model_b", "winner_tie"]].idxmax(axis=1)
print("Label distribution: all rows vs rows with a bad response_a / response_b")
print(pd.DataFrame({
    "all": label.value_counts(normalize=True),
    "bad_resp_a": label[df["response_a_bad"].apply(len) > 0].value_counts(normalize=True),
    "bad_resp_b": label[df["response_b_bad"].apply(len) > 0].value_counts(normalize=True),
}).round(3))

            rows_with_bad  None_turns  empty_turns  rows_where_last_turn_bad
prompt                  5           0            5                         5
response_a            133         120           31                        86
response_b            143         126           35                        91 

Rows with any bad turn: 209 (0.36%)

Label distribution: all rows vs rows with a bad response_a / response_b
                  all  bad_resp_a  bad_resp_b
winner_model_a  0.349       0.278       0.455
winner_model_b  0.342       0.398       0.252
winner_tie      0.309       0.323       0.294


### Where does the length come from?
#### Per side (A/B), in characters: whole dialog vs longest single turn (prompt_i + response_i).

In [ ]:
def clean(lst):
    return [t if isinstance(t, str) else "" for t in lst]

def side_lengths(prompts, resps):
    p, r = clean(prompts), clean(resps)
    turn_len = [len(pi) + len(ri) for pi, ri in zip(p, r)]
    return sum(turn_len), max(turn_len)

for side in ["a", "b"]:
    res = df.apply(lambda r: side_lengths(r["prompt_json"], r[f"response_{side}_json"]), axis=1)
    df[f"total_len_{side}"] = res.apply(lambda x: x[0])
    df[f"max_turn_len_{side}"] = res.apply(lambda x: x[1])

df["n_turns"] = df["prompt_json"].apply(len)
multi = df["n_turns"] > 1

q = [0.5, 0.75, 0.9, 0.95, 0.99]
print(df[["total_len_a", "max_turn_len_a", "total_len_b", "max_turn_len_b"]].quantile(q).round(0), "\n")

print("Median total_len_a | 1-turn rows:", df.loc[~multi, "total_len_a"].median(),
      "| multi-turn rows:", df.loc[multi, "total_len_a"].median())
print("Median max_turn_len_a | multi-turn rows:", df.loc[multi, "max_turn_len_a"].median(), "\n")

ratio = (df["max_turn_len_a"] / df["total_len_a"].clip(lower=1))[multi]
print("max_turn / total (side A, multi-turn rows):")
print(ratio.describe().round(3))

      total_len_a  max_turn_len_a  total_len_b  max_turn_len_b
0.50       1264.0          1197.0       1272.0          1205.0
0.75       2114.0          1926.0       2128.0          1937.0
0.90       3330.0          2759.0       3352.0          2746.0
0.95       4607.0          3508.0       4619.0          3506.0
0.99       9792.0          6694.0       9788.0          6685.0 

Median total_len_a | 1-turn rows: 1153.0 | multi-turn rows: 2705.0
Median max_turn_len_a | multi-turn rows: 1459.0 

max_turn / total (side A, multi-turn rows):
count    7539.000
mean        0.550
std         0.180
min         0.044
25%         0.434
50%         0.541
75%         0.648
max         1.000
dtype: float64


### Token length
### Per side: whole dialog vs longest single turn, in DeBERTa tokens, and the share of rows above 512 / 1024 tokens (special tokens not counted).

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

def n_tokens(texts, batch=2048):
    out = []
    for i in range(0, len(texts), batch):
        ids = tok(texts[i:i + batch], add_special_tokens=False)["input_ids"]
        out.extend(len(x) for x in ids)
    return out

flat = pd.DataFrame({
    "row": df.index.repeat(df["n_turns"]),
    "prompt": [t for l in df["prompt_json"] for t in clean(l)],
    "resp_a": [t for l in df["response_a_json"] for t in clean(l)],
    "resp_b": [t for l in df["response_b_json"] for t in clean(l)],
})

for c in ["prompt", "resp_a", "resp_b"]:
    flat["tok_" + c] = n_tokens(flat[c].tolist())
flat["tok_turn_a"] = flat["tok_prompt"] + flat["tok_resp_a"]
flat["tok_turn_b"] = flat["tok_prompt"] + flat["tok_resp_b"]

g = flat.groupby("row")
res = pd.DataFrame({
    "total_a": g["tok_turn_a"].sum(), "max_turn_a": g["tok_turn_a"].max(),
    "total_b": g["tok_turn_b"].sum(), "max_turn_b": g["tok_turn_b"].max(),
})

print(res.quantile([0.5, 0.75, 0.9, 0.95, 0.99]).round(0), "\n")
for lim in [512, 1024]:
    print(f"Share of rows > {lim} tokens:")
    print((res > lim).mean().round(3).to_string(), "\n")

/Users/vladimir/Projects/LLM-finetuning-competition/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


      total_a  max_turn_a  total_b  max_turn_b
0.50    275.0       261.0    276.0       263.0
0.75    455.0       418.0    456.0       418.0
0.90    737.0       595.0    737.0       593.0
0.95   1022.0       787.0   1031.0       786.0
0.99   2217.0      1532.0   2218.0      1556.0 

Share of rows > 512 tokens:
total_a       0.197
max_turn_a    0.147
total_b       0.198
max_turn_b    0.147 

Share of rows > 1024 tokens:
total_a       0.050
max_turn_a    0.024
total_b       0.051
max_turn_b    0.025 

